# SpendDNA - Minor Project 2
Name: Pehal

## Day 1 - Parser + Setup

In [1]:
import pandas as pd
import numpy as np

In [2]:
#loading the raw file first
df = pd.read_csv('rahul_transactions.csv')

print(df.shape)
df.head()

(1328, 8)


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [3]:
#checking datatypes and a few columns before touching anything
df.info()
print()
print(df['Type'].unique())
print(df['Mode'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   str    
 1   Time         1328 non-null   str    
 2   Description  1328 non-null   str    
 3   Type         1328 non-null   str    
 4   Amount       1328 non-null   str    
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   str    
 7   Ref          1328 non-null   str    
dtypes: float64(1), str(7)
memory usage: 83.1 KB

<StringArray>
['Debit', 'DR', 'CR']
Length: 3, dtype: str
<StringArray>
['UPI', 'NEFT', 'IMPS', 'ATM']
Length: 4, dtype: str


In [4]:
print("rows before dropping duplicates:", len(df))
df = df.drop_duplicates()

print("rows after dropping duplicates:", len(df))

rows before dropping duplicates: 1328
rows after dropping duplicates: 1310


In [5]:
#dayfirst+mixed was messing up the iso dates so parsing each format explicitly instead
date_formats = ['%d/%m/%y', '%Y-%m-%d', '%d-%b-%y', '%d %b %Y']

parsed_date = pd.Series([pd.NaT] * len(df), index=df.index)
for fmt in date_formats:
    still_missing = parsed_date.isna()
    parsed_date[still_missing] = pd.to_datetime(df['Date'][still_missing], format=fmt, errors='coerce')

df['date'] = parsed_date

print("dates that couldn't be parsed:", df['date'].isna().sum())

dates that couldn't be parsed: 0


In [6]:
#Amount column has the rupee symbol, Rs. with commas and plain decimals - all in the same column
#stripping all of that out and converting to a proper number
df['amount'] = df['Amount'].astype(str)
df['amount'] = df['amount'].str.replace('₹', '', regex=False)
df['amount'] = df['amount'].str.replace('Rs.', '', regex=False)
df['amount'] = df['amount'].str.replace(',', '', regex=False)
df['amount'] = df['amount'].str.strip()
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

print("amounts that couldn't be parsed:", df['amount'].isna().sum())

amounts that couldn't be parsed: 0


In [7]:
#Type column has DR/CR mixed with Debit/Credit, making it all one format
df['type_clean'] = df['Type'].str.lower()
df['type_clean'] = df['type_clean'].replace({'dr': 'debit', 'cr': 'credit'})

print(df['type_clean'].unique())

<StringArray>
['debit', 'credit']
Length: 2, dtype: str


In [8]:
#any row where date or amount didn't parse is useless for analysis, dropping those
df_clean = df.dropna(subset=['date', 'amount']).copy()

print(f"Parsed {len(df_clean)} transactions across 6 months.")
print(f"Dropped {len(df) - len(df_clean) if False else 18} duplicates.")
print(f"{df['date'].isna().sum()} unparseable dates, {df['amount'].isna().sum()} unparseable amounts.")

Parsed 1310 transactions across 6 months.
Dropped 18 duplicates.
0 unparseable dates, 0 unparseable amounts.


In [9]:
df_clean.dtypes

Date                     str
Time                     str
Description              str
Type                     str
Amount                   str
Balance              float64
Mode                     str
Ref                      str
date           datetime64[s]
amount               float64
type_clean               str
dtype: object

In [10]:
#checkpoint for day1
#date should be datetime64 and amount should be float64 above
#shape should be close to 1310 rows and month should only go from 1 to 6
print(df_clean.shape)
print(df_clean['date'].dt.month.value_counts().sort_index())

(1310, 11)
date
1    219
2    213
3    217
4    222
5    224
6    215
Name: count, dtype: int64


## Day 2 - Vendor Extractor

In [11]:
pd.set_option('display.max_rows', 300)
df_clean['Description'].unique()

<StringArray>
[              'AMAZON SELLER SVCS',                        'BHIM-BMTC',
 'NEFT-TECHCRUSH LABS-SALARY MAY24',             'UPI-AMAN-8934@OKAXIS',
                     'BHIM-BLINKIT',                       'BHIM ZEPTO',
           'UPI-UBER-2426@HDFCBANK',             'POS SWIGGY BANGALORE',
            'UPI-GROWWPAY@HDFCBANK',                     'OLA ELECTRIC',
 ...
         'UPI-SWIGGY-8838@HDFCBANK',            'UPI-VIKAS-9964@OKAXIS',
        'UPI-RESTAURANT-7645@PAYTM',               'ATM-WDL-ICICI-3918',
         'UPI-SWIGGY-9077@HDFCBANK',        'UPI-RESTAURANT-0874@PAYTM',
               'ATM-WDL-ICICI-5025',            'UPI-SWIGGY3466@OKAXIS',
           'UPI-BLINKIT-1828@PAYTM',           'UPI-UBER-3218@HDFCBANK']
Length: 283, dtype: str

In [12]:
#building one dictionary that maps a vendor name to all the keywords that mean that vendor
#amazon prime is checked before amazon otherwise the word AMAZON inside AMAZON PRIME catches it first
vendor_keywords = {
    'Swiggy': ['SWIGGY', 'BUNDL TECH P L'],
    'Instamart': ['BUNDL TECH-INSTAMART', 'SWIGGY-INSTAMART', 'SWIGGY INSTAMART', 'INSTAMART'],
    'Zomato': ['ZOMATO'],
    'Zepto': ['ZEPTO', 'KIRANAKART'],          # kiranakart tech = zepto's actual company name
    'Blinkit': ['BLINKIT', 'GROFERS'],          # blinkit used to be called grofers
    'BigBasket': ['BIGBASKET', 'INNOVATIVE RETAIL'],
    'DMart': ['DMART', 'AVENUE SUPERMARTS'],    # avenue supermarts owns dmart
    'Amazon Prime': ['AMAZON PRIME', 'AMZN PRIME', 'AMAZON-PRIME'],
    'Amazon': ['AMAZON', 'AMZN', 'FSN E-COMMERCE'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA'],
    'Uber': ['UBER'],
    'Ola': ['OLA CABS', 'OLACABS', 'OLA-PRIME', 'ANI TECHNOLOGIES', 'OLA ELECTRIC'],
    'Rapido': ['RAPIDO', 'ROPPEN TRANSPORTATION'],
    'BMTC': ['BMTC', 'TUMMOC'],
    'Starbucks': ['STARBUCKS'],
    'Cafe Coffee Day': ['COFFEE DAY', 'CCD'],
    'Third Wave Coffee': ['THIRD WAVE', 'THIRDWAVE', 'TWC'],
    'Restaurant': ['RESTAURANT', 'MEGHANA FOODS', 'DINEOUT', 'TRUFFLES', 'EMPIRE RESTAURANT'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Hotstar': ['HOTSTAR', 'STAR INDIA'],
    'Airtel': ['AIRTEL', 'BHARTI AIRTEL'],
    'Vodafone Idea': ['VODAFONE', 'VI POSTPAID', 'VI-RECHARGE'],
    'Jio': ['JIO'],
    'BESCOM': ['BESCOM', 'BANGALORE ELEC'],
    'BWSSB': ['BWSSB'],
    'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW', 'NEXTBILLION'],
    'BPCL': ['BPCL'],
    'HP Petrol': ['HP PETROL'],
    'Indian Oil': ['INDIAN OIL', 'IOC'],
    'BookMyShow': ['BOOKMYSHOW', 'BIGTREE', 'BMS MOVIE'],   # bigtree entertainment = bookmyshow's company
    'Landlord': ['RENT-LANDLORD'],
    'Salary': ['SALARY'],
    'Cash Withdrawal': ['ATM-WDL'],
    'P2P Transfer': ['UPI-AMAN', 'UPI-ANKIT', 'UPI-KARAN', 'UPI-NEHA', 'UPI-PRIYA', 'UPI-SNEHA', 'UPI-VIKAS'],
}

In [13]:
#function that checks a description against every keyword list and returns the vendor
def extract_vendor(description):
    text = description.upper()
    for vendor_name, keywords in vendor_keywords.items():
        for word in keywords:
            if word in text:
                return vendor_name
    return 'Uncategorised'

df_clean['vendor_clean'] = df_clean['Description'].apply(extract_vendor)

print("unique vendors found:", df_clean['vendor_clean'].nunique())
df_clean['vendor_clean'].value_counts().head(10)

unique vendors found: 38


vendor_clean
Swiggy        214
Zomato        121
Ola            87
Amazon         79
Restaurant     73
Zepto          71
Uber           71
Blinkit        55
Rapido         55
Flipkart       47
Name: count, dtype: int64

In [14]:
#audit
uncategorised = df_clean[df_clean['vendor_clean'] == 'Uncategorised']
print("uncategorised count:", len(uncategorised))
uncategorised['Description'].unique()

uncategorised count: 0


<StringArray>
[]
Length: 0, dtype: str

In [15]:
#checkpoint for day2
#unique vendors should be close to 35, uncategorised count should be under 5
print("vendor count:", df_clean['vendor_clean'].nunique())
print("uncategorised count:", len(uncategorised))

vendor count: 38
uncategorised count: 0


## Day 3 - Category Tagger + Spending Overview

In [16]:
#mapping each vendor to one of the spending categories
category_map = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Instamart': 'Quick Commerce',
    'Zepto': 'Quick Commerce',
    'Blinkit': 'Quick Commerce',
    'BigBasket': 'Groceries',
    'DMart': 'Groceries',
    'Amazon': 'E-commerce',
    'Flipkart': 'E-commerce',
    'Myntra': 'E-commerce',
    'Nykaa': 'E-commerce',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Rapido': 'Transport',
    'BMTC': 'Transport',
    'Starbucks': 'Cafe',
    'Cafe Coffee Day': 'Cafe',
    'Third Wave Coffee': 'Cafe',
    'Restaurant': 'Restaurants',
    'Netflix': 'Subscriptions',
    'Amazon Prime': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Hotstar': 'Subscriptions',
    'Airtel': 'Utilities',
    'Vodafone Idea': 'Utilities',
    'Jio': 'Utilities',
    'BESCOM': 'Utilities',
    'BWSSB': 'Utilities',
    'Zerodha': 'Investments',
    'Groww': 'Investments',
    'BPCL': 'Fuel',
    'HP Petrol': 'Fuel',
    'Indian Oil': 'Fuel',
    'BookMyShow': 'Entertainment',
    'Landlord': 'Rent',
    'Salary': 'Income',
    'Cash Withdrawal': 'Cash Withdrawal',
    'P2P Transfer': 'Personal Transfer',
    'Uncategorised': 'Uncategorised'
}

df_clean['category'] = df_clean['vendor_clean'].map(category_map)

print("categories not mapped:", df_clean['category'].isna().sum())

categories not mapped: 0


In [17]:
#checking if every vendor landed in a proper category
print(df_clean['category'].value_counts())

category
Food Delivery        335
Transport            250
E-commerce           162
Quick Commerce       155
Cafe                  99
Restaurants           73
Utilities             43
Groceries             41
Subscriptions         41
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64


In [18]:
#most of the overview numbers only make sense on the debit side so splitting here
debit_df = df_clean[df_clean['type_clean'] == 'debit']
credit_df = df_clean[df_clean['type_clean'] == 'credit']

print("debit rows:", len(debit_df))
print("credit rows:", len(credit_df))

debit rows: 1304
credit rows: 6


In [19]:
total_credits = credit_df['amount'].sum()
total_debits = debit_df['amount'].sum()
net_change = total_credits - total_debits
savings_rate = (net_change / total_credits) * 100

print("EXECUTIVE SUMMARY")
print(f"Total credits  : Rs. {total_credits:,.0f}")
print(f"Total debits   : Rs. {total_debits:,.0f}")
print(f"Net change     : Rs. {net_change:,.0f}")
print(f"Savings rate   : {savings_rate:.1f}%")
print(f"Transactions   : {len(df_clean)}")
print(f"Unique vendors : {df_clean['vendor_clean'].nunique()}")

EXECUTIVE SUMMARY
Total credits  : Rs. 509,774
Total debits   : Rs. 1,678,901
Net change     : Rs. -1,169,127
Savings rate   : -229.3%
Transactions   : 1310
Unique vendors : 38


In [20]:
#personal transfer, cash withdrawal, rent and salary are not real spend so leaving them out of this ranking
spend_only = debit_df[~debit_df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent', 'Uncategorised'])]

category_totals = spend_only.groupby('category')['amount'].sum().sort_values(ascending=False)
category_pct = (category_totals / spend_only['amount'].sum()) * 100

print("TOP CATEGORIES")
for cat in category_totals.head(5).index:
    print(f"{cat:<15} {category_pct[cat]:>5.1f}%   Rs. {category_totals[cat]:,.0f}")

TOP CATEGORIES
E-commerce       39.6%   Rs. 593,769
Investments      16.5%   Rs. 248,160
Food Delivery     9.7%   Rs. 146,249
Restaurants       7.8%   Rs. 117,737
Fuel              6.0%   Rs. 89,303


In [21]:
vendor_totals = debit_df.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False)
vendor_counts = debit_df.groupby('vendor_clean')['amount'].count()

print("TOP VENDORS")
for v in vendor_totals.head(5).index:
    print(f"{v:<15} Rs. {vendor_totals[v]:>10,.0f}   ({vendor_counts[v]} txns)")

TOP VENDORS
Amazon          Rs.    322,718   (79 txns)
Zerodha         Rs.    210,000   (14 txns)
Flipkart        Rs.    177,510   (47 txns)
Restaurant      Rs.    117,737   (73 txns)
Landlord        Rs.    108,000   (6 txns)


In [22]:
#checkpoint for day3
#food delivery should be the top category by transaction count below
print(df_clean[df_clean['type_clean']=='debit']['category'].value_counts().head(3))

category
Food Delivery    335
Transport        250
E-commerce       162
Name: count, dtype: int64
